# Cvičení 2: Neuronové sítě v Keras – Predikce cen nemovitostí (King County Housing)

> **Cíl cvičení:** Vytvořit regresní model neuronové sítě v knihovně **TensorFlow / Keras** pro predikci cen domů (`price`).
> Načíst předzpracovaný dataset `kc_house_data_preprocessed.csv`, normalizovat data, rozdělit na trénovací a testovací sadu,
> navrhnout architekturu sítě s omezením počtu neuronů ve skrytých vrstvách ($\le$ počet nezávislých proměnných, tj. $\le 18$),
> otestovat různé počty vrstev, zkompilovat s optimalizátorem **Adam** a ztrátou **MSE**, natrénovat a vyhodnotit metriky **MAE** a **RMSE** na testovací sadě.

---

## 1. Import knihoven a načtení dat

Načteme dataset `kc_house_data_preprocessed.csv` pomocí knihovny Pandas a zkontrolujeme strukturu.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

# Načtení předzpracovaného datasetu
data_path = "data/kc_house_data_preprocessed.csv"
df = pd.read_csv(data_path)
if "Unnamed: 0" in df.columns:
    df = df.drop(columns=["Unnamed: 0"])

print(f"Rozměry datasetu: {df.shape}")
df.head()

## 2. Příprava dat a rozdělení na Train / Test

Oddělíme cílovou proměnnou `price` od prediktorů a rozdělíme data na trénovací a testovací sadu (poměr 80:20).

In [ ]:
X = df.drop("price", axis=1)
y = df["price"]
n_features = X.shape[1]

print(f"Počet nezávislých proměnných (vstupních neuronů): {n_features}")
print(f"Názvy příznaků: {list(X.columns)}")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

print(f"Trénovací vzorky: {len(X_train):,}")
print(f"Testovací vzorky:  {len(X_test):,}")

## 3. Normalizace dat (StandardScaler)

Neuronové sítě jsou extrémně citlivé na měřítko vstupních proměnných (např. `sqft_lot` v řádu statisíců vs. `bathrooms` v jednotkách).
Proto nezávislé proměnné $X$ i cílovou proměnnou $y$ standardizujeme na nulový průměr a jednotkový rozptyl.

In [ ]:
# Normalizace vstupních příznaků
scaler_X = StandardScaler()
X_train_scaled = scaler_X.fit_transform(X_train)
X_test_scaled = scaler_X.transform(X_test)

# Škálování cílové proměnné pro stabilní gradientní sestup
scaler_y = StandardScaler()
y_train_scaled = scaler_y.fit_transform(y_train.values.reshape(-1, 1)).flatten()

print("Data byla úspěšně standardizována!")

## 4. Definice architektury neuronové sítě

V souladu se zadáním:
- Vstupní vrstva má 18 neuronů (počet nezávislých proměnných),
- Skryté vrstvy mají počet neuronů $\le 18$,
- Výstupní vrstva má 1 neuron s lineární aktivací (`activation='linear'`).

Otestujeme postupně se zužující architekturu se 3 skrytými vrstvami: $18 \to 12 \to 6 \to 1$.

In [ ]:
model = Sequential([
    Dense(18, activation="relu", input_shape=(n_features,), name="hidden_1"),
    Dense(12, activation="relu", name="hidden_2"),
    Dense(6, activation="relu", name="hidden_3"),
    Dense(1, activation="linear", name="output_price")
])

model.summary()

## 5. Kompilace modelu

Použijeme optimalizátor **Adam** a ztrátovou funkci **Mean Squared Error (MSE)**.

In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.005),
    loss="mean_squared_error",
    metrics=["mean_absolute_error"]
)
print("Model byl úspěšně zkompilován!")

## 6. Trénování modelu s validační sadou

Nastavíme `epochs=40`, `batch_size=128` a `validation_split=0.15`.

In [ ]:
history = model.fit(
    X_train_scaled,
    y_train_scaled,
    epochs=40,
    batch_size=128,
    validation_split=0.15,
    verbose=1
)

## 7. Predikce a vyhodnocení na testovací sadě (MAE a RMSE)

Predikujeme na `X_test_scaled`, provedeme inverzní transformaci na původní měřítko v USD a spočteme MAE a RMSE.

In [ ]:
# Predikce a inverzní transformace
y_pred_scaled = model.predict(X_test_scaled)
y_pred = scaler_y.inverse_transform(y_pred_scaled).flatten()

# Výpočet metrik
test_mae = mean_absolute_error(y_test, y_pred)
test_rmse = np.sqrt(mean_squared_error(y_test, y_pred))
test_r2 = r2_score(y_test, y_pred)

print("=== VÝSLEDKY NA TESTOVACÍ SADĚ (King County Housing MLP) ===")
print(f"Průměrná absolutní chyba (MAE): {test_mae:,.2f} USD")
print(f"Odmocnina střední kvadratické chyby (RMSE): {test_rmse:,.2f} USD")
print(f"Koeficient determinace R²: {test_r2*100:.2f} %")

## 8. Vizualizace křivek ztrátové funkce (Loss)

Zobrazíme průběh trénovací a validační MSE ztráty v závislosti na epoše.

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(history.history["loss"], label="Trénovací MSE")
plt.plot(history.history["val_loss"], label="Validační MSE", linestyle="--")
plt.title("Vývoj ztrátové funkce MSE během trénování")
plt.xlabel("Epocha")
plt.ylabel("Loss (MSE)")
plt.legend()
plt.grid(True)
plt.show()